In [82]:
from ngsolve import *
from ngsolve.webgui import Draw
import ipywidgets as widgets
from netgen.occ import *
import numpy as np
from IPython.display import clear_output, display

In [83]:
# geometry from the assignment picture
L = 2.2
H = 0.41
xc, yc = 0.2, 0.2
R = 0.05

shape = Rectangle(L, H).Circle(xc, yc, R).Reverse().Face()
shape.edges.name = "wall"
shape.edges.Min(X).name = "inlet"
shape.edges.Max(X).name = "outlet"

Draw(shape)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3…

BaseWebGuiScene

In [84]:
maxh = 0.03   # try 0.02 for finer mesh
mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=maxh)).Curve(3)
Draw(mesh)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

BaseWebGuiScene

In [85]:
# physical parameters
rho = 1.0      # constant density
nu = 0.01      # viscosity
D = 2*R        # cylinder diameter = 0.1

# choose Reynolds number here
Re = 50.0

# characteristic inlet velocity from Re = U*D/nu
Umean = Re * nu / D
Umax = 1.5 * Umean

print("Re   =", Re)
print("nu   =", nu)
print("rho  =", rho)
print("Umean=", Umean)
print("Umax =", Umax)

Re   = 50.0
nu   = 0.01
rho  = 1.0
Umean= 5.0
Umax = 7.5


In [86]:
V = VectorH1(mesh, order=3, dirichlet="wall|inlet")
Q = H1(mesh, order=2)
X = V * Q

u, p = X.TrialFunction()
v, q = X.TestFunction()

In [87]:
stokes = (nu * InnerProduct(grad(u), grad(v))
          + div(u) * q
          + div(v) * p
          - 1e-10 * p * q) * dx

a = BilinearForm(stokes).Assemble()
f = LinearForm(X).Assemble()

gfu = GridFunction(X)

In [88]:
uin = CoefficientFunction((Umax * 4 * y * (H - y) / (H * H), 0))
gfu.components[0].Set(uin, definedon=mesh.Boundaries("inlet"))

Draw(gfu.components[0], mesh, "vel")

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

BaseWebGuiScene

In [89]:
inv_stokes = a.mat.Inverse(X.FreeDofs())

res = f.vec - a.mat * gfu.vec
gfu.vec.data += inv_stokes * res

Draw(gfu.components[0], mesh)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

BaseWebGuiScene

In [90]:
""" tau = 0.001   # reduce if unstable, e.g. 0.001 or 0.0005

mstar = BilinearForm(u * v * dx + tau * stokes).Assemble()
inv = mstar.mat.Inverse(X.FreeDofs(), inverse="sparsecholesky")

conv = BilinearForm(X, nonassemble=True)
conv += (Grad(u) * u) * v * dx """

' tau = 0.001   # reduce if unstable, e.g. 0.001 or 0.0005\n\nmstar = BilinearForm(u * v * dx + tau * stokes).Assemble()\ninv = mstar.mat.Inverse(X.FreeDofs(), inverse="sparsecholesky")\n\nconv = BilinearForm(X, nonassemble=True)\nconv += (Grad(u) * u) * v * dx '

In [91]:
""" t = 0.0
i = 0
tend = 15.0

gfut = GridFunction(V, multidim=0)
vel = gfu.components[0]

scene = Draw(gfu.components[0], mesh, "velocity",
             min=0, max=max(1.2*Umax, 1e-6), autoscale=False)

tw = widgets.Text(value='t = 0.0')
display(tw)

with TaskManager():
    while t < tend:
        res = conv.Apply(gfu.vec) + a.mat * gfu.vec
        gfu.vec.data -= tau * inv * res

        t += tau
        i += 1
        
        if i % 25 == 0:
            gfut.AddMultiDimComponent(vel.vec)
        if i % 50 == 0:
            tw.value = f"t = {t:.3f}"

    tw.value = f"t = {t:.3f}" """

' t = 0.0\ni = 0\ntend = 15.0\n\ngfut = GridFunction(V, multidim=0)\nvel = gfu.components[0]\n\nscene = Draw(gfu.components[0], mesh, "velocity",\n             min=0, max=max(1.2*Umax, 1e-6), autoscale=False)\n\ntw = widgets.Text(value=\'t = 0.0\')\ndisplay(tw)\n\nwith TaskManager():\n    while t < tend:\n        res = conv.Apply(gfu.vec) + a.mat * gfu.vec\n        gfu.vec.data -= tau * inv * res\n\n        t += tau\n        i += 1\n\n        if i % 25 == 0:\n            gfut.AddMultiDimComponent(vel.vec)\n        if i % 50 == 0:\n            tw.value = f"t = {t:.3f}"\n\n    tw.value = f"t = {t:.3f}" '

In [96]:
tau = 0.001

# mass matrix on velocity only
m = BilinearForm(X)
m += u * v * dx
m.Assemble()

# implicit part: mass + tau * Stokes
mstar = BilinearForm(X)
mstar += u * v * dx + tau * stokes
mstar.Assemble()

inv = mstar.mat.Inverse(X.FreeDofs(), inverse="sparsecholesky")

# explicit convection
conv = BilinearForm(X, nonassemble=True)
conv += (Grad(u) * u) * v * dx


In [98]:
t = 0.0
i = 0
tend = 15.0

vel = gfu.components[0]
vort = grad(vel)[1,0] - grad(vel)[0,1]

tw = widgets.Text(value='t = 0.0')
display(tw)

with TaskManager():
    while t < tend:
        res = conv.Apply(gfu.vec) + a.mat * gfu.vec
        gfu.vec.data -= tau * inv * res

        # enforce inlet data every step
        gfu.components[0].Set(uin, definedon=mesh.Boundaries("inlet"))

        t += tau
        i += 1

        if i % 50 == 0:
            tw.value = f"t = {t:.3f}"
            clear_output(wait=True)
            display(tw)
            Draw(vort, mesh, "vorticity", min=-50, max=50, autoscale=False)

tw.value = f"t = {t:.3f}"

Text(value='t = 15.000')

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…